## 1. Initialize the project

Create the working context for the Project, if it has not already been created. The project serves as a placeholder for the code, data, and management of data operations inside the platform.

In [ ]:
import digitalhub as dh
PROJECT_NAME = "<YOUR_PROJECT_NAME>"
proj = dh.get_or_create_project(PROJECT_NAME)

Note: Make sure to replace <YOUR_PROJECT_NAME> with the actual name of your project before running the code.

## Prerequisites

Before running the dataset creation and fine-tuning jobs, create a Hugging Face access token and store it as a project secret named `HF_TOKEN`.

1. Open [Hugging Face Settings → Access Tokens](https://huggingface.co/settings/tokens).
2. Generate a new token.
3. Assign the required permissions for model and dataset access.
4. Copy the token value.
5. In the current project, create a secret named `HF_TOKEN` and paste the token as its value.

> The jobs below use `HF_TOKEN` to authenticate against Hugging Face. Do not hardcode the token in the notebook or source files.

## 2. Deploy Training Function


Fetch the "train" operation in the project. Train function is a python job that performs LLM fine tuning from a predefined HuggingFace model and dataset.

In [ ]:
function_train = proj.get_function("train-llm")

Build the function as shown below.

In [ ]:
func_b = function_train.run(action='build',wait=True)

## 3. Perform Training
Training parameters define the model and dataset to be used. Training run refers to HF_TOKEN secret and expects a volume to be created of an appropriate size.

In [ ]:
train_run = function_train.run(action="job",
                     profile="1xV100",
                     parameters={
                         "model_id": "meta-llama/Llama-3.1-8B",
                         "model_name": "llmpa-tracked",
                         "hf_dataset_name": "team-bay/data-science-qa",
                         "train_data_path": "DataScienceDataset.csv",
                         "dev_data_path": "DataScienceDataset.csv",
                         "num_epochs": 2
                     },
                     secrets=["HF_TOKEN"],
                     envs=[
                        {"name": "HF_HOME", "value": "/app/local_data/huggingface"},
                        {"name": "TRANSFORMERS_CACHE", "value":  "/app/local_data/huggingface"}
                     ],
                     volumes=[{
                        "volume_type": "persistent_volume_claim",
                        "name": "volume-llmpa",
                        "mount_path": "/app/local_data",
                        "spec": { "size": "50Gi" }}]
					)